In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)


In [2]:
def simulate_season():
    days = 120

    # Crop parameters
    LAI_max = np.random.uniform(4.5, 6.0)
    r = np.random.uniform(0.06, 0.10)
    RUE = np.random.uniform(1.3, 1.6)
    HI = np.random.uniform(0.32, 0.42)

    # Persistent seasonal stress
    stress = np.random.uniform(0.7, 1.0)

    LAI = 0.3
    biomass = 0.0

    for _ in range(days):
        PAR = np.random.normal(18, 2)
        PAR = max(PAR, 5)

        LAI = LAI + r * LAI * (1 - LAI / LAI_max) * stress
        LAI = min(LAI, LAI_max)

        IPAR = PAR * (1 - np.exp(-0.6 * LAI))
        dB = RUE * IPAR * stress
        biomass += dB

    yield_kg_ha = biomass * HI
    return yield_kg_ha, stress, LAI_max, r, RUE, HI


In [3]:
N_SAMPLES = 10000
YIELD_MIN = 500
YIELD_MAX = 3000


In [4]:
records = []
rejected = 0

while len(records) < N_SAMPLES:
    y, stress, LAI_max, r, RUE, HI = simulate_season()

    if YIELD_MIN <= y <= YIELD_MAX:
        records.append({
            "yield_kg_per_ha": y,
            "seasonal_stress": stress,
            "LAI_max": LAI_max,
            "leaf_growth_rate": r,
            "RUE": RUE,
            "harvest_index": HI
        })
    else:
        rejected += 1

len(records), rejected


(10000, 69)

In [5]:
df_synthetic = pd.DataFrame(records)
df_synthetic.head()


,yield_kg_per_ha,seasonal_stress,LAI_max,leaf_growth_rate,RUE,harvest_index
0,723.141829,0.746806,5.061810,0.098029,1.519598,0.379866
1,801.607688,0.950591,5.448459,0.085341,1.460732,0.329029
2,841.708845,0.898059,4.734656,0.070010,1.464768,0.391460
3,910.010770,0.860829,4.616602,0.098976,1.595863,0.389816
4,693.478102,0.823119,4.760060,0.095871,1.324070,0.372451


In [6]:
df_synthetic.describe()


,yield_kg_per_ha,seasonal_stress,LAI_max,leaf_growth_rate,RUE,harvest_index
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,757.374592,0.852216,5.241694,0.080026,1.450545,0.370442
std,124.030652,0.086176,0.436413,0.011562,0.086607,0.028695
min,500.299263,0.700001,4.500010,0.060000,1.300001,0.320003
25%,663.408178,0.778185,4.862842,0.069950,1.375903,0.345744
50%,747.968102,0.853311,5.237152,0.080085,1.451471,0.370936
75%,844.380208,0.926445,5.622650,0.089962,1.524779,0.395318
max,1161.727978,0.999982,5.999697,0.099994,1.599953,0.419999


In [7]:
df_synthetic["yield_kg_per_ha"].quantile([0.1, 0.5, 0.9])


0.1    599.511149
0.5    747.968102
0.9    927.372066
Name: yield_kg_per_ha, dtype: float64

In [8]:
df_synthetic.to_csv(
    "synthetic_groundnut_digital_twin_dataset.csv",
    index=False
)
